# Chapter 18 — File-system workflows: pathlib, CSV, JSON, and import metadata

**Book role:** Data workflows  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use pathlib and explicit file encodings.
- Read and write CSV/JSON safely.
- Record import provenance and file metadata.
- Build a source-adapter contract that reports schema and failures.

## 1. Pathlib and file selection

Use `Path` for path composition, suffix checks, directory creation, and existence tests. Avoid hard-coded paths tied to one laptop. A GUI file dialog should produce a selected path; the file service then decides whether the extension and content are supported. Extensions are hints, not proof that a file contains the expected format.

A robust file workflow separates selection, validation, reading, normalization, and presentation. For example, Open CSV selects a path; the import service reads it with a declared encoding; schema validation checks required columns and types; the cleaning pipeline records issues; finally the view displays a summary. If reading fails, preserve the last successfully loaded dataset unless the user explicitly requested replacement.

### Worked example 1: Read CSV with explicit policy

This intentionally checks for the expected file path before loading. In an individual chapter notebook, adjust the path to the course root if the current working directory is different.

In [ ]:
from pathlib import Path
import pandas as pd

path = Path("data") / "sample_sales.csv"
if not path.exists():
    raise FileNotFoundError(f"Expected sample file at {path.resolve()}")
sales = pd.read_csv(path, encoding="utf-8", low_memory=False)
print("Rows:", len(sales))
print("Columns:", sales.columns.tolist())
print(sales.head())

## 2. CSV and JSON trade-offs

CSV is widely interoperable and human-readable, but it does not preserve rich dtypes, multiple sheets, formatting, or nested structures. Values are interpreted during import, which means date formats, leading zeros in IDs, decimal separators, and missing-value markers need explicit policy. JSON supports structured values and nested objects, but inconsistent nested schemas often need normalization before analysis.

Always decide the encoding and error policy. `utf-8-sig` can help with CSV exported by some spreadsheet tools that include a UTF-8 byte-order mark. Avoid silently replacing every decode error with a replacement character when data accuracy matters; log it and ask for a valid export where needed. When writing, set a stable column order and newline policy and write atomically when partial output would be harmful.

### Worked example 2: Write JSON records and an import manifest

This demonstrates a manifest structure; a real import can include a file hash, validation counts, and transformation version. Do not assume JSON is always the best exchange format for a DataFrame.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
records_path = output_dir / "preview.json"
manifest_path = output_dir / "import_manifest.json"
preview = pd.DataFrame({"order_id": [1, 2], "region": ["North", "West"]})
preview.to_json(records_path, orient="records", indent=2, date_format="iso")
manifest = {
    "source": "inline demonstration",
    "imported_at_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": len(preview),
    "columns": preview.columns.tolist(),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote:", records_path, manifest_path)

## 3. Import provenance and repeatability

An import log should capture the source filename, import timestamp, input row count, columns, delimiter/encoding settings where relevant, detected schema version, transformations applied, rejected-row count, and output dataset identity. A hash can help identify the exact bytes imported, though it does not prove the content is trustworthy. Do not preserve private source files by default just to have a provenance record; follow data retention and privacy policy.

A source adapter should return predictable information: a DataFrame (or iterable), source metadata, warnings, and validation issues. The UI can then summarize the operation. This common interface makes CSV, Excel, MySQL, and MongoDB fit the same import workflow without claiming that all sources have identical transactional semantics.

## Guided lab: Lab: source import adapter

Implement `load_csv(path) -> ImportResult` where `ImportResult` contains the DataFrame, source metadata, warnings, and validation issues. Reject unsupported formats explicitly; do not infer that every `.csv` has comma separators or UTF-8 encoding. Test missing files, empty files, BOM-containing CSV, duplicate headers, malformed rows, and IDs that contain leading zeros.

## Review questions and practice

1. Why is a filename extension not enough to establish file validity?
2. What does CSV fail to preserve compared with Excel?
3. What metadata helps reproduce an import?
4. Why should a failed import not necessarily discard the previously loaded dataset?

## Further reading

- pathlib: https://docs.python.org/3/library/pathlib.html
- pandas CSV I/O: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- Python JSON: https://docs.python.org/3/library/json.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.